In [1]:
#import env and load them into the environment variables
import os
from langchain_groq import ChatGroq
from dotenv import load_dotenv
load_dotenv()
groq_api_key = os.getenv("GROQ_API_KEY")
    
_llm = ChatGroq(
    model="llama-3.1-8b-instant",
    groq_api_key=groq_api_key
)


c:\Users\sivan_7\OneDrive\Desktop\rag\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#invoke the llm with a prompt
response = _llm.invoke("What is the capital of France?")
print(response)

content='The capital of France is Paris.' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 42, 'total_tokens': 50, 'completion_time': 0.004426744, 'completion_tokens_details': None, 'prompt_time': 0.045611177, 'prompt_tokens_details': None, 'queue_time': 0.047999953, 'total_time': 0.050037921}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_f757f4b0bf', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019cdb33-a904-7f10-9d68-f0de2ffec4cc-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 42, 'output_tokens': 8, 'total_tokens': 50}


In [4]:
#here we need to upload a document and then retrieve it using the vector retriever and the bm25 retriever. We will then merge the results and return the final answer to the user.
from pathlib import Path
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import (
    PyPDFLoader,
    Docx2txtLoader,
    TextLoader,
    CSVLoader,
)


In [13]:
def _unwrap(docs: list) -> list[Document]:
    """
    Some loaders return (Document, score) tuples instead of plain Documents.
    This unwraps them safely.
    """
    clean = []
    for item in docs:
        if isinstance(item, tuple):
            item = item[0]          # unwrap (Document, anything) → Document
        if isinstance(item, Document):
            clean.append(item)
    return clean

In [19]:
def load_documents(file_paths: list[str]) -> list[Document]:
    all_docs: list[Document] = []
    print(f"Loading documents from: {file_paths}")
    for path_str in file_paths:
        path = Path(path_str)
        print(f"Processing file: {path}")
        if not path.exists():
            raise FileNotFoundError(f"File not found: {path_str}")

        ext = path.suffix.lower()

        if ext == ".pdf":
            raw = PyPDFLoader(str(path)).load()
            #print(f"-------------------{raw}--------------------")  # Debug: print raw loaded content
        elif ext == ".docx":
            raw = Docx2txtLoader(str(path)).load()
        elif ext in (".txt", ".text", ".md", ".markdown"):
            raw = TextLoader(str(path), encoding="utf-8").load()
        elif ext == ".csv":
            raw = CSVLoader(str(path)).load()
        else:
            raise ValueError(
                f"Unsupported file type '{ext}'. "
                "Supported: .pdf, .docx, .txt, .csv, .md"
            )
    #print(f"Loaded {len(raw)} raw documents from {path.name}-----{raw[0]}...")
#load_documents(["Agentic-RAG/requirements.txt"])

        raw = _unwrap(raw)   # ← unwrap tuples right after loading
        print(f"Loaded {len(raw)} raw documents from {path.name}")
        for doc in raw:
            doc.metadata.setdefault("source", str(path))

        all_docs.extend(raw)
        print(f"Total documents loaded so far: {len(all_docs)}")
        CHUNK_SIZE = 1000
        CHUNK_OVERLAP = 300
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=300,
        separators=["\n\n", "\n", " ", ""]
     )
    chunked = _unwrap(splitter.split_documents(all_docs))  # ← unwrap after splitting too
    print(f"Split into {len(chunked)} chunks (chunk size={CHUNK_SIZE}, overlap={CHUNK_OVERLAP})")
    return chunked
docs=load_documents(["C:\\Users\\sivan_7\\OneDrive\\Desktop\\rag\\Agentic-RAG\\langgraphflow\\MAHESH - CV.pdf"])


Loading documents from: ['C:\\Users\\sivan_7\\OneDrive\\Desktop\\rag\\Agentic-RAG\\langgraphflow\\MAHESH - CV.pdf']
Processing file: C:\Users\sivan_7\OneDrive\Desktop\rag\Agentic-RAG\langgraphflow\MAHESH - CV.pdf
Loaded 1 raw documents from MAHESH - CV.pdf
Total documents loaded so far: 1
Split into 8 chunks (chunk size=1000, overlap=300)


Up to here we just loaded AND chunked the doc

In [40]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever


In [36]:
#here we need to build embeddings for the documents and then store them in the vector database. 
def _ensure_documents(results: list) -> list[Document]:
    """
    Normalise retriever output — some retrievers return (Document, score)
    tuples instead of plain Document objects. This unwraps them safely.
    """
    clean = []
    for item in results:
        if isinstance(item, tuple):
            # (Document, score) tuple — take the Document
            doc = item[0]
        else:
            doc = item
        if hasattr(doc, "page_content"):
            clean.append(doc)
        else:
            #logger.warning("[hybrid_search] Skipping unexpected item type: %s", type(item))
            print(f"Skipping unexpected item type: {type(item)}")
    return clean
TOP_K = 5
embeddings   = HuggingFaceEmbeddings(model="BAAI/bge-base-en-v1.5")
vector_store     = FAISS.from_documents(docs, embeddings)
vector_retriever = vector_store.as_retriever(
        search_type="similarity",
        search_kwargs={"k": TOP_K},
    )
bm25_retriever = BM25Retriever.from_documents(docs)
bm25_retriever.k = TOP_K
#vector_results = _ensure_documents(vector_retriever.invoke(query))
#here we need to see how the doc was stored i vectorstore
#print(f"Vector store has {len(vector_store.index)} documents.")



Loading weights: 100%|██████████| 199/199 [00:00<00:00, 799.86it/s]
BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
print("Total vectors stored:", vector_store.index.ntotal)
for doc_id, doc in vector_store.docstore._dict.items():
    print("Doc ID:", doc_id)
    print("Content:", doc.page_content[:200])  # first 200 chars
    print("Metadata:", doc.metadata)
    print("-"*50)

In [ ]:
# here we need to do semantic search and BM25 search and then merge the results and return the final answer to the user.
QUESTION = "What is the experience of Mahesh?"

BM25_WEIGHT = 0.25
VECTOR_WEIGHT = 0.75
def safe_content(item) -> str:
    """
    Safely extract page_content from a Document or a (Document, score) tuple.
    Never crashes — always returns a string.
    """
    if isinstance(item, tuple):
        item = item[0]
    if hasattr(item, "page_content"):
        return item.page_content
    return str(item)

def hybrid_search_node(bm25_weight=BM25_WEIGHT, vector_weight=VECTOR_WEIGHT, QUESTION=QUESTION, top_k=TOP_K):
    # This is a placeholder for your merging logic.
    # You could, for example, combine the results based on relevance scores,
    # or simply concatenate them and remove duplicates.
    vector_retriver=vector_retriever.invoke(QUESTION)
    vector_results = _ensure_documents(vector_retriver)
    bm25_retriver=bm25_retriever.invoke(QUESTION)
    bm25_results = _ensure_documents(bm25_retriver)
    #COMBINE THE RESULTS with conditions
    combined_results = vector_results + bm25_results
    # Remove duplicates while preserving order
    seen = set()
    unique_results = []
    for result in combined_results:
        key = safe_content(result)
        if key not in seen:
            seen.add(key)
            unique_results.append(result)
    return unique_results[:top_k]
final_results = hybrid_search_node()
print(f"Final combined results (top {TOP_K}):")
for i, result in enumerate(final_results, start=1):
    print(f"{i}. {safe_content(result)}")
#here we need to validate similarity search results and bm25 search results how they contributed



In [ ]:
#upto here RAG is working fine
#here we need to build the graph
